# Native Apps Framework

Build and distribute full applications (data + logic + UI) within Snowflake. Apps run inside the consumer's account with controlled access via privilege grants.

**Key concepts:**
- **Application Package** — A container (extended database) holding data content, setup scripts, manifest, and version metadata.
- **manifest.yml** — Required metadata file defining setup script path, readme, and app configuration.
- **setup_script.sql** — SQL that runs automatically when a consumer installs the app; creates schemas, views, procedures, and grants.
- **Application Roles** — Database-role-like constructs scoped to the app, used to grant consumers access to specific objects.
- **Versioned Schema** — Special schema type that prevents upgrades from interfering with concurrent code execution.

| Provider Action | Consumer Action |
|---|---|
| Create application package | Install app from listing |
| Upload code + data to stage | Grant requested privileges |
| Add version & set release directive | Use app objects (views, procs) |
| Publish listing via Provider Studio | Upgrade when new version available |

## 1. Setup — Provider Stage & Data

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE DATABASE native_app_demo;
CREATE SCHEMA native_app_demo.provider_data;
CREATE SCHEMA native_app_demo.app_code;
CREATE OR REPLACE STAGE native_app_demo.app_code.v1_stage;

USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS app_admin;
CREATE ROLE IF NOT EXISTS share_monitor;
GRANT ROLE app_admin TO ROLE SYSADMIN;
GRANT ROLE share_monitor TO ROLE app_admin;
GRANT USAGE ON DATABASE native_app_demo TO ROLE app_admin;
GRANT USAGE ON ALL SCHEMAS IN DATABASE native_app_demo TO ROLE app_admin;
GRANT CREATE VIEW ON SCHEMA native_app_demo.provider_data TO ROLE app_admin;
GRANT IMPORTED PRIVILEGES ON DATABASE snowflake TO ROLE share_monitor;

-- Sample data
USE ROLE ACCOUNTADMIN;
CREATE TABLE native_app_demo.provider_data.sales AS
SELECT o_orderkey AS order_id, o_custkey AS customer_id,
       o_totalprice AS amount, o_orderdate AS order_date,
       o_orderstatus AS status
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS
LIMIT 10000;

GRANT SELECT ON ALL TABLES IN SCHEMA native_app_demo.provider_data TO ROLE app_admin;

## 2. Create Application Package

In [ ]:
CREATE APPLICATION PACKAGE IF NOT EXISTS demo_analytics_app_pkg
    COMMENT = 'Analytics app with sales insights';

GRANT USAGE ON DATABASE native_app_demo TO APPLICATION PACKAGE demo_analytics_app_pkg;
GRANT USAGE ON SCHEMA native_app_demo.provider_data TO APPLICATION PACKAGE demo_analytics_app_pkg;
GRANT SELECT ON TABLE native_app_demo.provider_data.sales TO APPLICATION PACKAGE demo_analytics_app_pkg;

## 3. Application Files

The app needs: `manifest.yml` (metadata), `setup_script.sql` (object creation in consumer), and optionally `README.md`.

### manifest.yml
```yaml
manifest_version: 1
artifacts:
  setup_script: setup_script.sql
  readme: README.md
```

### setup_script.sql
```sql
CREATE APPLICATION ROLE IF NOT EXISTS app_viewer;
CREATE OR ALTER VERSIONED SCHEMA app_content;

CREATE OR REPLACE SECURE VIEW app_content.v_sales_summary AS
SELECT DATE_TRUNC('month', order_date) AS month,
       COUNT(*) AS order_count,
       SUM(amount) AS total_revenue
FROM native_app_demo.provider_data.sales
GROUP BY month;

GRANT USAGE ON SCHEMA app_content TO APPLICATION ROLE app_viewer;
GRANT SELECT ON ALL VIEWS IN SCHEMA app_content TO APPLICATION ROLE app_viewer;
```

> **Note:** In production, these files are uploaded to the stage via `PUT` or the Snowflake CLI. This notebook shows the content for reference.

## 4. Upload Files to Stage

In [ ]:
-- Upload files to stage (run from SnowSQL or Snowflake CLI):
-- PUT file:///path/to/manifest.yml @native_app_demo.app_code.v1_stage/;
-- PUT file:///path/to/setup_script.sql @native_app_demo.app_code.v1_stage/;
-- PUT file:///path/to/README.md @native_app_demo.app_code.v1_stage/;

-- Verify stage contents
LIST @native_app_demo.app_code.v1_stage;

## 5. Add Version

In [ ]:
-- Add version from stage (requires files on stage)
-- ALTER APPLICATION PACKAGE demo_analytics_app_pkg
--     ADD VERSION v1_0 USING '@native_app_demo.app_code.v1_stage';

-- Set release directive
-- ALTER APPLICATION PACKAGE demo_analytics_app_pkg
--     SET DEFAULT RELEASE DIRECTIVE VERSION = v1_0 PATCH = 0;

SHOW APPLICATION PACKAGES LIKE 'DEMO_ANALYTICS_APP_PKG';

## 6. Install Application (Local Testing)

In [ ]:
-- Install the app locally for testing
-- CREATE APPLICATION demo_analytics_app
--     FROM APPLICATION PACKAGE demo_analytics_app_pkg;

-- Verify application
-- SHOW APPLICATIONS LIKE 'DEMO_ANALYTICS_APP';

## 7. Consumer Privilege Grants

When a consumer installs the app, they grant privileges the app requests:
```sql
-- Consumer grants:
-- GRANT USAGE ON WAREHOUSE <wh> TO APPLICATION demo_analytics_app;
```

## 8. Distribution

In [ ]:
-- Set distribution for external sharing
-- ALTER APPLICATION PACKAGE demo_analytics_app_pkg SET DISTRIBUTION = EXTERNAL;

-- Then create a listing via Provider Studio in Snowsight
DESCRIBE APPLICATION PACKAGE demo_analytics_app_pkg;

## 9. Teardown

In [ ]:
USE ROLE ACCOUNTADMIN;
-- DROP APPLICATION IF EXISTS demo_analytics_app;
DROP APPLICATION PACKAGE IF EXISTS demo_analytics_app_pkg;
DROP DATABASE IF EXISTS native_app_demo;
DROP ROLE IF EXISTS app_admin;
DROP ROLE IF EXISTS share_monitor;

## References

- [Native App Framework Overview](https://docs.snowflake.com/en/developer-guide/native-apps/native-apps-about)
- [CREATE APPLICATION PACKAGE](https://docs.snowflake.com/en/sql-reference/sql/create-application-package)
- [CREATE APPLICATION](https://docs.snowflake.com/en/sql-reference/sql/create-application)
- [Manifest Reference](https://docs.snowflake.com/en/developer-guide/native-apps/manifest-reference)
- [Setup Script](https://docs.snowflake.com/en/developer-guide/native-apps/creating-setup-script)
- [Tutorial: Create a Basic Native App](https://docs.snowflake.com/en/developer-guide/native-apps/tutorials/getting-started-tutorial)
- [Share Data Content in a Native App](https://docs.snowflake.com/en/developer-guide/native-apps/sharing-data-content)